# TD 4 — Evaluating an LLM 📏

A model "looks good". But **how good**, and how can we know? In this lab, you will measure instead of judging by eye.

**At the end of this lab, you will know how to:**

- build a reference annotated by hand;
- compute precision, recall and the F1 score;
- compare prompts and models;
- measure the agreement between two people;
- recognise the traps: small samples, misleading accuracy, tuning on the test data.

We use the task of TD 3 again: extract the places from a travel story and say whether they were visited.

## Set up the environment ⚙️

This lab works with a **local** model (Ollama, see TD 2) or a **hosted** one (Gemini, see TD 1). Choose in the next cell. Packages needed: `pydantic pandas scikit-learn`, + `ollama` / `google-genai python-dotenv`.

The `ask` function sends a prompt and returns the text of the answer. If you give it a Pydantic schema, the answer is JSON.

> A local model with 3 billion parameters makes mistakes: this is normal, and interesting to observe. Do not use FunctionGemma here, it is not made for reading texts.

In [ ]:
import os
import json
from pathlib import Path
from time import sleep

BACKEND = "ollama"            # "ollama" (local) or "gemini" (hosted)
MODELS = {"ollama": "qwen2.5:3b", "gemini": "gemini-3.5-flash-lite"}

def ask(prompt, schema=None, model=None, backend=None):
    """Send a prompt, return the text of the answer (JSON if a schema is given)."""
    backend = backend or BACKEND
    model = model or MODELS[backend]

    if backend == "ollama":
        import ollama
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            format=schema.model_json_schema() if schema else None,
            options={"temperature": 0, "num_ctx": 8192, "num_predict": 3000},
        )
        return response.message.content

    from dotenv import load_dotenv
    from google import genai
    from google.genai import errors, types
    load_dotenv()
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    config = types.GenerateContentConfig(
        response_mime_type="application/json" if schema else None,
        response_schema=schema,
    )
    for attempt in range(4):
        try:
            return client.models.generate_content(model=model, contents=prompt, config=config).text
        except errors.APIError as error:
            if error.code not in (429, 500, 503) or attempt == 3:
                raise
            sleep(2 ** attempt)

print(ask("Say hello in French, in three words."))

## 1. Build the reference ✍️

To measure the errors of a model, we first need the **right answers**. They are called the reference (*gold standard*), and a human being writes them.

We will use two stories:

- `voyage_vaucluse.txt` to **develop** our prompts;
- `voyage_bouches_du_rhone.txt` to **measure** the final result (test). Do not look at the errors on this story before the end!

**Annotation guide.** Two people must be able to annotate in the same way:

- annotate the **named** places: towns, villages, neighbourhoods, streets, monuments, natural sites, rivers;
- do not annotate departments, or generic words ("le port", "l'église", "les arènes");
- each place is annotated only once, without its article;
- the status is one of: `visited` (the traveller went there), `seen` (seen from a distance), `considered` (thought about, but not done), `mentioned` (simply named).

In [ ]:
DEV, TEST = "voyage_vaucluse.txt", "voyage_bouches_du_rhone.txt"
stories = {name: Path("data/texts", name).read_text(encoding="utf-8") for name in (DEV, TEST)}

print(stories[DEV])

❓ **>>> Exercise 1.** Work **alone**, without a model. Read both stories and create a dictionary `gold` like this:

```python
gold = {
    DEV: {"Avignon": "visited", ...},
    TEST: {"Marseille": "visited", ...},
}
```

Write down the cases where you hesitated.

## 2. Measure 📐

We compare the set of places **predicted** by the model with the set of places in the **reference**:

- **TP** (true positives): places predicted and present in the reference;
- **FP** (false positives): places predicted by mistake;
- **FN** (false negatives): places forgotten.

$$\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN} \qquad F_1 = \frac{2 \times \text{precision} \times \text{recall}}{\text{precision} + \text{recall}}$$

- Precision answers: "among what the model found, how much is right?"
- Recall answers: "among what had to be found, how much was found?"

To compare names, we **normalise** them: lower case, no accents, no article.

In [ ]:
import re
import unicodedata

def normalize(name):
    name = unicodedata.normalize("NFD", name.lower().replace("’", "'"))
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")
    name = name.replace("-", " ").strip()
    name = re.sub(r"^(le |la |les |l')", "", name)
    return re.sub(r"\s+", " ", name)

normalize("L'Isle-sur-la-Sorgue")

❓ **>>> Exercise 2.** Write a function `evaluate(predicted, reference)` that takes two lists of names and returns a dictionary `{"precision": ..., "recall": ..., "f1": ...}`. Beware of divisions by zero.

Check it **by hand** with this small example: the reference contains Nice, Menton, Grasse and Tende; the model found nice, Menton and Paris.

## 3. Compare prompts 🧪

Here are the output schema and a first prompt, simple on purpose.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ValidationError

class Place(BaseModel):
    name: str
    visit_status: Literal["visited", "seen", "considered", "mentioned"]

class Places(BaseModel):
    places: list[Place]

PROMPTS = {"A_simple": "Extract the places from this text:"}

def extract(prompt, text, **options):
    """Return a dictionary {name: status}, empty if the answer is invalid."""
    answer = ask(f"{prompt}\n\nText:\n{text}", schema=Places, **options)
    try:
        return {p.name: p.visit_status for p in Places.model_validate_json(answer).places}
    except ValidationError:
        print("Invalid answer")
        return {}

prediction = extract(PROMPTS["A_simple"], stories[DEV])
prediction

❓ **>>> Exercise 3.** Evaluate prompt A on the **development** story. Also display the list of false positives and the list of forgotten places. What are the causes of error?

❓ **>>> Exercise 4.** Add two prompts to the `PROMPTS` dictionary:

- `B_rules`: the rules of the annotation guide;
- `C_examples`: the same rules **and** a short example invented by you, with its answer.

Compare the three prompts on the **development** story, in a pandas `DataFrame`. You can improve them as much as you want, but **only** with this story.

❓ **>>> Exercise 5.** It is **test** time. Choose your best prompt from the development story, then measure it **only once** on the test story. If you can, compare two models (for example `qwen2.5:3b` and `qwen2.5:1.5b`, or Ollama and Gemini).

Is the test score better or worse? Why should you not go back and change the prompt now?

## 4. The status: beware of accuracy 🎭

Now let's look at `visit_status`. **Accuracy** is the share of right answers. It is easy to compute... and easy to misread.

❓ **>>> Exercise 6.** On the test story, and only for the places found by both the model and the reference:

1. compute the accuracy of the status;
2. compute the accuracy of a **dumb model** that always answers `visited`;
3. display a cross table (reference / model) with `pd.crosstab`.

Is the model better than the dumb model?

## 5. What about humans? 🤝

If two people do not agree, we cannot ask the model to be perfect. We measure the agreement with **Cohen's kappa**, which removes the agreement due to chance:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

- $p_o$: share of cases where both people give the same answer;
- $p_e$: agreement expected by chance, from how often each person gives each answer.

`1`: perfect agreement. `0`: no better than chance.

❓ **>>> Exercise 7.**

1. Write a function `kappa(a, b)` that takes two lists of statuses. Check it with `sklearn.metrics.cohen_kappa_score` on the lists below.
2. Exchange your `gold` dictionary with a classmate. Compute the kappa on the places you both annotated.
3. Discuss your disagreements: should one of you be corrected, or should the **guide be made clearer**?

```python
a = ["visited", "visited", "seen", "considered", "visited", "seen"]
b = ["visited", "seen",    "seen", "considered", "visited", "considered"]
```

## 6. How good is our measure? 🎲

Our test contains about twenty places in **a single** story. With so little data, a difference of a few points means nothing. A simple way to see it is the ***bootstrap***: draw at random, **with replacement**, as many places as in the reference, compute the recall again, and repeat 1,000 times.

❓ **>>> Exercise 8.** For each place of the test reference, write `1` if it was found and `0` otherwise. Make 1,000 draws, then display the recall and the interval that contains 95% of the values (`np.percentile`). Are two models with recalls of 0.75 and 0.80 really different?

## Summary 🧾

Fill in this form to make your experiment **reproducible**:

| Item | Your answer |
| --- | --- |
| Model and version | |
| Date | |
| Parameters | |
| Exact prompt | |
| Development and test data | |
| Precision, recall, F1 (with the uncertainty) | |
| Main errors | |

## Discussion 💭

- For your project, is good recall or good precision better? Give an example of each case.
- Some people use **an LLM to evaluate another LLM**. What are the advantages and the risks?
- The stories may have been written with the help of an AI, or seen during the training of the model. What would that change?